# Relatório Mensal de Investimentos — Estratégia de Dividendos

**Como usar:**
1. Adicione sua chave Gemini nos Secrets (ícone de cadeado): `GEMINI_API_KEY`
2. Execute todas as células: `Ctrl+F9` ou `Runtime → Run all`
3. O arquivo `.docx` será salvo em `Meu Drive/relatorios_besst/`

**O que este relatório gera:**
- Ranking TOP 10 com gráfico comparativo P/L e P/VP
- Gráfico de evolução de dividendos das TOP 5 (últimos 5 anos)
- Tabela de Oportunidades Ocultas TOP 5


In [ ]:
# CÉLULA 1 — Instalação das bibliotecas
print('Instalando bibliotecas...')
import subprocess
subprocess.run(['pip', 'install', '-q',
    'python-docx', 'matplotlib', 'yfinance'], check=True)
print('Instalado!')

In [ ]:
# CÉLULA 2 — Conexão com o Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
PASTA_DRIVE = '/content/drive/MyDrive/relatorios_besst'
os.makedirs(PASTA_DRIVE, exist_ok=True)
print(f'Drive conectado! Relatórios em: {PASTA_DRIVE}')

In [ ]:
# CÉLULA 3 — Chave Gemini (não usada neste relatório,
# mas mantida para compatibilidade futura)
from google.colab import userdata
import os
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print(f'Chave carregada: {GEMINI_API_KEY[:6]}...{GEMINI_API_KEY[-4:]}')
except Exception:
    print('Chave nao configurada — ok para este relatório.')
    GEMINI_API_KEY = None

In [ ]:
# CÉLULA 4 — Baixar e carregar o script do GitHub
# Substitua pela URL raw do seu repositório
GITHUB_RAW_URL = 'https://raw.githubusercontent.com/queirozcleiton/relatorio_dividendos_besst/main/relatorio_mensal_besst.py'

import urllib.request, traceback

req = urllib.request.Request(
    GITHUB_RAW_URL,
    headers={'Authorization': 'token SEU_TOKEN_AQUI'}  # remova se repositório for público
)
try:
    with urllib.request.urlopen(req) as response:
        codigo = response.read().decode('utf-8')
except Exception:
    # Tenta sem token (repositório público)
    with urllib.request.urlopen(GITHUB_RAW_URL) as response:
        codigo = response.read().decode('utf-8')

print(f'Script baixado! ({len(codigo.splitlines())} linhas)')

# Remove bloco __main__ e executa no escopo global
linhas_filtradas = []
dentro_main = False
for linha in codigo.splitlines(keepends=True):
    if linha.strip().startswith('if __name__') and '__main__' in linha:
        dentro_main = True
    if not dentro_main:
        linhas_filtradas.append(linha)

try:
    exec(''.join(linhas_filtradas), globals())
    print('Funcoes carregadas!')
    ok = all(f in globals() for f in [
        'buscar_ibovespa','buscar_fundamentus',
        'aplicar_filtros_ranking','aplicar_filtros_oportunidades',
        'calcular_ranking','buscar_historico_dividendos_anual',
        'gerar_relatorio_mensal'
    ])
    print(f'Todas as funções disponíveis: {ok}')
except Exception:
    traceback.print_exc()

In [ ]:
# CÉLULA 5 — Criar pastas e executar o pipeline completo
from google.colab import files as colab_files
import shutil, os, math, time

criar_pastas()
os.makedirs(PASTA_DRIVE, exist_ok=True)

from datetime import datetime
inicio = datetime.now()
print('=== RELATORIO MENSAL — CARTEIRA DE DIVIDENDOS ===')

print('\nColetando dados...')
tickers_ibov = buscar_ibovespa()
time.sleep(1)
df_todos = buscar_fundamentus()

print('\nAplicando filtros do ranking...')
df_ranking_filtrado = aplicar_filtros_ranking(df_todos, tickers_ibov)
df_ranking = calcular_ranking(df_ranking_filtrado)

# Passa o TOP 10 para excluir da Seção 3
tickers_top10 = df_ranking.head(TOP_RANKING).index.tolist()
print('\nBuscando tickers BESST fora do TOP 10...')
df_oo = aplicar_filtros_oportunidades(df_todos, tickers_top10)

# Histórico de dividendos (TOP 5 do ranking)
tickers_hist = df_ranking.head(TOP_HISTORICO).index.tolist()
print(f'\nHistórico de dividendos para: {tickers_hist}')
historico_divs, df_hist_ind, anos_ref = buscar_historico_dividendos_anual(tickers_hist)

# Ordena Seção 3 por P/VP crescente
df_oo = df_oo.copy()
df_oo['cagr_div'] = df_oo.index.map(
    lambda t: df_hist_ind.loc[t, 'cagr_div']
    if t in df_hist_ind.index else float('nan')
)
df_oo = df_oo.sort_values('p/vp', ascending=True, na_position='last')

print('\nGerando relatório Word com gráficos...')
caminho = gerar_relatorio_mensal(
    df_ranking, df_oo, historico_divs, df_hist_ind, anos_ref
)

agora_str = datetime.now().strftime('%Y%m')
nome = f'relatorio_mensal_besst_{agora_str}.docx'
caminho_drive = os.path.join(PASTA_DRIVE, nome)
caminho_local = f'/content/{nome}'
shutil.copy(caminho, caminho_drive)
shutil.copy(caminho, caminho_local)

tempo = (datetime.now() - inicio).seconds
print(f'\nRelatório gerado em {tempo}s!')
print(f'Drive: {caminho_drive}')
colab_files.download(caminho_local)